# Student starter
Run cells in order. An incomplete TODO prints guidance and stops subsequent cells. After implementing it, restart the kernel and run again.

# Exercises 8–9 — Attention and Pretrained Transformers

Static word vectors → contextual information → scores → weights → context → Transformer tasks. Shared source provides all algorithms. Exactly TODOs 8.1–8.3; Exercise 9 is experimentation. This notebook remains in working/reference form.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        from pathlib import Path
        import sys
        root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
        if not (root / "src").exists():
            raise RuntimeError("Run from repository root or notebooks directory.")
        if str(root) not in sys.path:
            sys.path.insert(0, str(root))
        import numpy as np
        from src.transformers_nlp import (calculate_attention_scores, softmax_attention_weights,
            calculate_weighted_context, attention_example, analyze_sentiment, recognize_entities, answer_from_context)
        from src.transformer_resources import MODELS, load_transformer_pipeline, TransformerModelUnavailable
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## Static context problem and numeric example

`bank approved the loan` versus `students sat on the river bank`: Word2Vec gives bank one vector. Attention combines surrounding representations. **Simplified educational attention example:** these invented vectors are not learned BERT/GPT weights. Query asks for information; keys advertise features; values contribute content. Predict each dot product before running.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        query = [1, 0]
        keys = [[1, 0], [0, 1], [0.5, 0.5]]
        values = [[2, 0], [0, 2], [1, 1]]
        scores = calculate_attention_scores(query, keys)  # TODO 8.1
        print("Scores:", scores)
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## Softmax — TODO 8.2

Subtract max before exp to preserve ratios and avoid overflow. Normalize the whole vector. Weights sum to one; extreme floating-point gaps can round tiny weights to zero. Calculate one weight on paper.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        weights = softmax_attention_weights(scores)
        print("Weights:", weights, "Sum:", weights.sum())
        print("Constant shift unchanged:", np.allclose(weights, softmax_attention_weights(scores + 10000)))
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## Weighted context — TODO 8.3

Multiply each value row by its weight and add the rows. Calculate one coordinate on paper. Change only values: do the weights change?

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        context_vector = calculate_weighted_context(weights, values)
        print("Weighted rows:", weights[:, None] * np.asarray(values))
        print("Final context:", context_vector)
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## Self-attention, scaling and multiple heads

Self-attention derives Q/K/V from tokens in the same sequence using learned W_Q/W_K/W_V. Each attention-matrix row is a token attending across other tokens. `softmax(QKᵀ / sqrt(d_k))V` scales larger-dimensional dot products before softmax. Multiple heads use different learned projections; no guaranteed grammar/coreference labels. Attention weights do not fully explain reasoning.

Language analogy: `The student submitted the assignment because it was due.` Context can relate it to assignment; toy vectors do not prove model coreference behavior.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        scaled = attention_example(query, keys, values, scaled=True)
        print("Scaled scores:", scaled["effective_scores"])
        print("Scaled weights/context:", scaled["weights"], scaled["context"])
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## Transformer architecture, BERT and GPT

Tokenizer → token embeddings + positions → stacked attention/feed-forward blocks with residual connections and normalization → task output.

BERT: **Bidirectional Encoder Representations from Transformers**, typically encoder-oriented. Masked language modeling (`submitted the [MASK] before the deadline`) builds contextual representations.

GPT: **Generative Pre-trained Transformer**, typically decoder-oriented and causal. Next-token prediction generates a continuation one token at a time. No GPT model is loaded.

| Feature | BERT | GPT |
|---|---|---|
| Typical architecture | Encoder | Decoder |
| Context | Bidirectional | Causal |
| Objective intuition | Masked token | Next token |
| Common use | Classification, NER, extractive QA | Generation |

MiniLM from Tasks 08–09 already used Transformer encoders and sentence-similarity training. Sentence vectors and task labels are different outputs of related technology.

## Exercise 9 — pretrained CPU tasks, no algorithm TODO

Before class: `python -m src.transformer_resources --download`. Loads below are local-only and cached, with no automatic downloads. Missing resources produce guidance; attention above remains usable. Scores are task estimates, not truth. We use pinned public Apache-2.0 DistilBERT variants, not vanilla BERT or GPT.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        pipelines = {}
        for task in MODELS:
            try:
                pipelines[task] = load_transformer_pipeline(task)
                print(task, MODELS[task]["id"], "CPU model ready")
            except TransformerModelUnavailable as error:
                print(error)
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


### Sentiment

Try positive, negative and mixed feedback. Labels do not capture every opinion, sarcasm or domain nuance.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        if "sentiment" in pipelines:
            for text in ["The NLP workshop was engaging and easy to follow.",
                         "The instructions were confusing and frustrating.",
                         "The examples were excellent but the instructions were confusing."]:
                print(text, analyze_sentiment(pipelines["sentiment"], text))
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


### Named Entity Recognition

Inspect category, original text, score and spans. The fictional institution is an educational example. Add location/person/course terms and observe misses or mislabels.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        entity_text = "Riya studies Natural Language Processing at Hindu College of Engineering."
        if "ner" in pipelines:
            entities = recognize_entities(pipelines["ner"], entity_text)
            print(entities)
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


### Extractive QA over supplied context — not RAG

The answer is a span from context. No document search or text generation. SQuAD-v1 training does not reliably teach abstention when an answer is absent; unsupported questions can still return incorrect spans.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        qa_context = "Natural Language Processing is offered in Semester 6. The course introduces text preprocessing, TF-IDF, word embeddings, attention, Transformers and semantic retrieval."
        if "qa" in pipelines:
            answer = answer_from_context(pipelines["qa"],
                "In which semester is Natural Language Processing offered?", qa_context)
            print(answer)
            print("Exact slice:", qa_context[answer["start"]:answer["end"]])
            unsupported = answer_from_context(pipelines["qa"], "What is the examination fee?", qa_context)
            print("Unsupported question — verify critically:", unsupported)
    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## Limitations, experiments and transition

Record predicted output → measured label/span and score → supported by input? Scores are not factual confidence. Change context locally, not canonical facts. Inputs ≤400 tokens (QA combined), questions ≤64; excessive input is rejected without chunking.

Pretrained knowledge can be incomplete/outdated and does not include private fictional documents automatically. Generative models can produce unsupported text; none is run here. What if we do not know which document supplies context? Retrieval can provide evidence later. Task 09 exposed full-document truncation: next extract and divide documents into meaningful units. No retrieval-to-QA, chunking, RAG or generation is implemented.

Exercise-specific Checkpoint 7 — Attention and Transformer NLP; original RAG Checkpoint 7 remains pending.